In [72]:
import pandas as pd


X_train = pd.read_csv("data/X_train.csv")
X_val   = pd.read_csv("data/X_val.csv")
X_test  = pd.read_csv("data/X_test.csv")

y_train = pd.read_csv("data/y_train.csv").values.ravel()
y_val   = pd.read_csv("data/y_val.csv").values.ravel()
y_test  = pd.read_csv("data/y_test.csv").values.ravel()

print(X_train.shape, y_train.shape)

(536, 11) (536,)


In [73]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [74]:
def evaluate(model, X, y):
    preds = model.predict(X)
    return {
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0)
    }

In [75]:
lr = LogisticRegression(max_iter=1000)

lr_params = {
    "C": [0.01, 0.1, 1, 10, 100]
}

lr_grid = GridSearchCV(lr, lr_params, cv=5, scoring="f1", n_jobs=-1)
lr_grid.fit(X_train, y_train)

lr_best = lr_grid.best_estimator_
print("LR VAL:", evaluate(lr_best, X_val, y_val))

LR VAL: {'accuracy': 0.7844827586206896, 'precision': 0.7222222222222222, 'recall': 0.6341463414634146, 'f1': 0.6753246753246753}


In [76]:
rf = RandomForestClassifier(random_state=42)

rf_params = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 5, 10, 20]
}

rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring="f1", n_jobs=-1)
rf_grid.fit(X_train, y_train)

rf_best = rf_grid.best_estimator_
print("RF VAL:", evaluate(rf_best, X_val, y_val))

RF VAL: {'accuracy': 0.7758620689655172, 'precision': 0.6829268292682927, 'recall': 0.6829268292682927, 'f1': 0.6829268292682927}


In [77]:
gb = GradientBoostingClassifier()

gb_params = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1]
}

gb_grid = GridSearchCV(gb, gb_params, cv=5, scoring="f1", n_jobs=-1)
gb_grid.fit(X_train, y_train)

gb_best = gb_grid.best_estimator_
print("GB VAL:", evaluate(gb_best, X_val, y_val))

GB VAL: {'accuracy': 0.7758620689655172, 'precision': 0.7419354838709677, 'recall': 0.5609756097560976, 'f1': 0.6388888888888888}


In [78]:
models = {
    "logistic_regression": lr_best,
    "random_forest": rf_best,
    "gradient_boosting": gb_best
}

best_model = None
best_name = None
best_f1 = -1

for name, model in models.items():
    preds = model.predict(X_val)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name

In [79]:

test_metrics = evaluate(best_model, X_test, y_test)

print("TEST RESULTS:",test_metrics)

TEST RESULTS: {'accuracy': 0.7758620689655172, 'precision': 0.6842105263157895, 'recall': 0.65, 'f1': 0.6666666666666666}


In [80]:
!pip install joblib
import joblib
joblib.dump(best_model, "models/best_model.pkl")

['models/best_model.pkl']

In [81]:
import json

metrics = {
    "test_metrics": test_metrics
}

with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

In [82]:
metadata = {
    "best_model": best_name,
    "model_type": type(best_model).__name__,
    "note": "Selected using validation F1 after GridSearchCV tuning"
}

with open("models/final_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)